# Structural Connectome C

EC2-native PhD analysis notebook for the ADNI structural connectome cohort.

This notebook restores the archived sectioned analysis, fixes the EC2 path bootstrap, and adds structural innovation analyses aligned with `SOTA_V_6.0.pptx`. While Step 7 post outputs are still rebuilding, all inferential outputs are labelled provisional until a final snapshot is locked.

## Research Objective Map

The notebook is organized around the current thesis objectives:

- Replicate established AD-spectrum findings in DTI microstructure and graph topology.
- Quantify connectome-wide structural disruption across CN, MCI, and AD.
- Separate short-range and long-range pathway integrity using connectome length and edge-distance relationship proxies.
- Test delay-proxy and structural compensation hypotheses without claiming functional decoupling until real FC / EEG / MEG matrices are available.
- Export reproducible tables and figures for dissertation, manuscript, and presentation use.

## 0. Setup and Paths

This cell binds the notebook to `/home/ec2-user/exp`, makes FSL/MRtrix binaries visible, imports all analysis modules, and creates an explicit provisional/final snapshot label.

In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import importlib
import os
import sys
import traceback

import numpy as np
import pandas as pd
from IPython.display import display, Markdown, Image

PROJECT_ROOT = Path.home() / "exp"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

for candidate in [
    Path.home() / "bin",
    Path.home() / "mrtrix3" / "bin",
    Path.home() / "fsl" / "bin",
    Path.home() / "fsl" / "share" / "fsl" / "bin",
]:
    if candidate.exists() and str(candidate) not in os.environ.get("PATH", "").split(os.pathsep):
        os.environ["PATH"] = f"{candidate}{os.pathsep}" + os.environ.get("PATH", "")

os.environ.setdefault("FSLDIR", str(Path.home() / "fsl"))

from connectome_pipeline.pipeline_paths import resolve_pipeline_paths

pipeline_paths = resolve_pipeline_paths(PROJECT_ROOT, create_layout=True)
print(pipeline_paths.format_summary())

# Main run policy. Keep this provisional while Step 7 post/connectomes are still rebuilding.
ANALYSIS_SNAPSHOT_MODE = "provisional"  # switch to "final" only after Step 7 post is locked
SNAPSHOT_UTC = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

# Section toggles. Heavy sections can be disabled independently during development.
analysis_config = True
analysis_cohort = True
analysis_dti = True
analysis_graph = True
analysis_edges = True
analysis_exports = True
analysis_brain_age = True
analysis_clinical = True
analysis_coupling = True
analysis_length_delay = True
analysis_advanced = True
analysis_functional_placeholder = True

REBUILD_MASTER = True
TOP_N_ROIS = 10
PRIMARY_CONNECTOME = "fd_sum"
LENGTH_CONNECTOME = "len_mean"
EDGEWISE_CONNECTOME_TYPE = "fd_sum"
EDGEWISE_N_PERM = 199
DELAY_VELOCITY_MM_PER_MS = 6.0

SECTION_RESULTS = {}
SECTION_STATUS = []
MAX_INLINE_FIGURES_PER_SECTION = 24
MAX_INLINE_TABLES_PER_SECTION = 8


def show_section_outputs(result, max_figures=MAX_INLINE_FIGURES_PER_SECTION, max_tables=MAX_INLINE_TABLES_PER_SECTION):
    if not isinstance(result, dict) or "out_dir" not in result:
        return
    out_dir = Path(result["out_dir"])
    if not out_dir.exists():
        return
    md_files = sorted(out_dir.glob("*.md"))
    for md_path in md_files[:4]:
        text = md_path.read_text(encoding="utf-8", errors="ignore").strip()
        if text:
            display(Markdown(f"**Inference: `{md_path.name}`**\n\n{text}"))
    table_patterns = ["*summary*.csv", "*pairwise.csv", "*node_tests.csv", "*components.csv", "*descriptives.csv"]
    table_files = []
    for pattern in table_patterns:
        table_files.extend(sorted(out_dir.glob(pattern)))
    seen = set()
    unique_tables = []
    for table_path in table_files:
        if table_path in seen:
            continue
        seen.add(table_path)
        unique_tables.append(table_path)
    for table_path in unique_tables[:max_tables]:
        try:
            table = pd.read_csv(table_path)
        except Exception as exc:
            display(Markdown(f"Could not display `{table_path.name}`: `{exc}`"))
            continue
        display(Markdown(f"**Stat table: `{table_path.name}`**"))
        display(table.head(15))
    pngs = sorted(out_dir.glob("*.png"))
    for png_path in pngs[:max_figures]:
        display(Markdown(f"**Figure: `{png_path.name}`**"))
        display(Image(filename=str(png_path)))
    if len(pngs) > max_figures:
        display(Markdown(f"Displayed {max_figures} of {len(pngs)} figures in `{out_dir}`; all figures are saved on disk."))


def run_section(name: str, enabled: bool, fn):
    if not enabled:
        row = {"section": name, "status": "disabled", "error": ""}
        SECTION_STATUS.append(row)
        display(Markdown(f"**{name}**: disabled"))
        return None
    try:
        result = fn()
    except FileNotFoundError as exc:
        row = {"section": name, "status": "skipped_missing_input", "error": str(exc)}
        SECTION_STATUS.append(row)
        display(Markdown(f"**{name}**: skipped, missing input: `{exc}`"))
        return None
    except ValueError as exc:
        msg = str(exc)
        if "No connectomes" in msg or "Not enough" in msg or "aligned" in msg:
            row = {"section": name, "status": "skipped_insufficient_input", "error": msg}
            SECTION_STATUS.append(row)
            display(Markdown(f"**{name}**: skipped, insufficient input: `{msg}`"))
            return None
        raise
    except Exception:
        row = {"section": name, "status": "error", "error": traceback.format_exc(limit=3)}
        SECTION_STATUS.append(row)
        raise
    SECTION_RESULTS[name] = result
    SECTION_STATUS.append({"section": name, "status": "ok", "error": ""})
    display(Markdown(f"**{name}**: ok"))
    show_section_outputs(result)
    return result


def maybe_display(obj, key=None, head=None):
    if obj is None:
        return
    data = obj.get(key) if isinstance(obj, dict) and key else obj
    if data is None:
        return
    if isinstance(data, pd.DataFrame):
        display(data.head(head) if head else data)
    else:
        display(data)


## Analysis Readiness

This is a real import smoke check. It verifies helper dependencies such as `analysis_plots.py` and `analysis_stats.py`, not only the presence of top-level files.

In [ ]:
MODULES = [
    "connectome_analysis.analysis_config",
    "connectome_analysis.analysis_stats",
    "connectome_analysis.analysis_plots",
    "connectome_analysis.analysis_cohort",
    "connectome_analysis.analysis_dti",
    "connectome_analysis.analysis_graph",
    "connectome_analysis.analysis_edges",
    "connectome_analysis.analysis_exports",
    "connectome_analysis.analysis_brain_age",
    "connectome_analysis.analysis_clinical",
    "connectome_analysis.analysis_coupling",
    "connectome_analysis.analysis_length_delay",
    "connectome_analysis.analysis_advanced",
    "connectome_analysis.analysis_live_connectome",
]

module_rows = []
loaded = {}
for name in MODULES:
    try:
        mod = importlib.import_module(name)
        mod = importlib.reload(mod)
        loaded[name] = mod
        module_rows.append({"module": name, "import_ok": True, "error": ""})
    except Exception as exc:
        module_rows.append({"module": name, "import_ok": False, "error": f"{type(exc).__name__}: {exc}"})

module_status = pd.DataFrame(module_rows)
display(module_status)
if not module_status["import_ok"].all():
    raise RuntimeError("Analysis module readiness failed; fix failed imports before running inference sections.")

from connectome_analysis.analysis_config import get_analysis_paths, apply_plot_theme
from connectome_analysis.analysis_cohort import load_master_cohort, qc_snapshot, run_data_completeness, run_demographics_overview
from connectome_analysis.analysis_dti import run_global_rd_analysis, run_local_rd_analysis, run_multimetric_dti_analysis
from connectome_analysis.analysis_graph import run_connectome_qc, run_global_graph_metrics, run_nodewise_metrics, run_centrality_review
from connectome_analysis.analysis_coupling import run_coupling_analysis
from connectome_analysis.analysis_edges import run_edgewise_inference
from connectome_analysis.analysis_brain_age import run_brain_age_analysis
from connectome_analysis.analysis_length_delay import run_lr_sr_analysis, run_delay_analysis, microstructure_delay_available
from connectome_analysis.analysis_clinical import run_clinical_covariate_review
from connectome_analysis.analysis_exports import write_excel_bundle, collect_csv_exports
from connectome_analysis.analysis_advanced import run_structural_innovation_analysis
from connectome_analysis.analysis_live_connectome import (
    run_live_global_microstructure,
    run_live_nodewise_microstructure,
    run_live_multimetric_overview,
    run_live_global_graph_metrics,
    run_live_nodewise_graph_metrics,
    run_live_structure_microstructure_coupling,
    run_live_brain_age,
)

apply_plot_theme()
paths = get_analysis_paths(
    notebook_dir=pipeline_paths.project_root,
    deriv_root=pipeline_paths.deriv_root,
    cohort_dti_csv=pipeline_paths.cohort_dti_csv,
    cohort_mri_csv=pipeline_paths.cohort_mri_csv,
)
paths


## Dataset Snapshot

This snapshot makes the moving state explicit. The analysis can run now, but manuscript-grade inference should be rerun after Step 7 post/connectomes are complete and QC-locked.

In [ ]:
snapshot = {
    "snapshot_utc": SNAPSHOT_UTC,
    "mode": ANALYSIS_SNAPSHOT_MODE,
    "project_root": str(pipeline_paths.project_root),
    "derivatives_root": str(paths.deriv_root),
    "connectomes_dir": str(paths.connectomes_dir),
    "analysis_output_root": str(paths.output_root),
}
connectome_counts = {}
if paths.connectomes_dir.exists():
    for suffix in ["ALL", "count", "fd_sum", "fa_mean", "md_mean", "len_mean"]:
        connectome_counts[f"n_{suffix}"] = len(list(paths.connectomes_dir.glob(f"SC_AAL_*_{suffix}.csv")))
snapshot.update(connectome_counts)
snapshot_df = pd.DataFrame([snapshot])
display(snapshot_df)
snapshot_df.to_csv(paths.output_root / "analysis_snapshot.csv", index=False)

try:
    from connectome_pipeline import pipeline_status
    pipeline_status = importlib.reload(pipeline_status)
    status = pipeline_status.display_group_stage_status(
        paths.deriv_root,
        cohort_dti_csv=paths.cohort_dti_csv,
        title="Pipeline group status (Connectome C snapshot)",
    )
    display(status)
except Exception as exc:
    display(Markdown(f"Pipeline status display skipped: `{type(exc).__name__}: {exc}`"))


## 1. Cohort Assembly and Subject Mapping

Builds the canonical master cohort table by merging DTI metadata, MRI metadata, available QC/metrics workbooks, and connectome availability flags. Groups are harmonized to `CN`, `MCI`, and `AD`.

In [ ]:
master = run_section(
    "01_cohort_master",
    analysis_cohort,
    lambda: load_master_cohort(paths, rebuild=REBUILD_MASTER),
)
maybe_display(master, head=10)
if master is None:
    raise RuntimeError("Master cohort could not be built; downstream sections require it.")
master_counts = master.groupby("group")["subject_id"].nunique().rename("n_subjects").reset_index()
display(master_counts)
qc_stage_counts = qc_snapshot(paths, master)
display(qc_stage_counts)


## 2. QC and Data Completeness

Separates cohort size, bias-corrected DWI availability, Step 7 prep/FOD/tracks/SIFT2/parcellation/DTI completion, and final connectome completion.

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)
QC_STRICT_TRACKS_COUNT = False
QC_TRACK_MINIMUM = 3_000_000
qc_out = run_section(
    "02_qc_data_completeness",
    analysis_cohort,
    lambda: run_data_completeness(
        paths,
        master,
        strict_tracks_count=QC_STRICT_TRACKS_COUNT,
        track_minimum=QC_TRACK_MINIMUM,
    ),
)
maybe_display(qc_out, "completeness")


## 3. Demographics and Clinical Overview

Summarizes age, sex, group counts, and available clinical scores with the same robust statistical framework used for imaging measures.

In [ ]:
demo_out = run_section(
    "03_demographics_clinical_overview",
    analysis_cohort,
    lambda: run_demographics_overview(paths, master),
)
maybe_display(demo_out, "counts")
maybe_display(demo_out, "clinical_summary")


## 4. Global DTI Analysis

Replicates the lead global RD analysis from the prior notebook and reports robust group statistics plus covariate-aware permutation sensitivity.

In [ ]:
rd_out = run_section(
    "04_global_dti_rd",
    analysis_dti,
    lambda: run_global_rd_analysis(paths, master),
)
if rd_out is None:
    rd_out = run_section(
        "04b_live_global_microstructure",
        analysis_dti,
        lambda: run_live_global_microstructure(paths, master),
    )
maybe_display(rd_out, "summary")


## 5. Local / ROI DTI Analysis

Tests regional RD parcel-wise with BH-FDR across ROIs and exports top regions for presentation review.

In [ ]:
local_rd_out = run_section(
    "05_local_roi_dti_rd",
    analysis_dti,
    lambda: run_local_rd_analysis(paths, master, top_n=TOP_N_ROIS),
)
if local_rd_out is None:
    local_rd_out = run_section(
        "05b_live_nodewise_microstructure",
        analysis_dti,
        lambda: run_live_nodewise_microstructure(paths, master, top_n=TOP_N_ROIS),
    )
maybe_display(local_rd_out, "summary")


## 6. Multi-metric DTI Analysis

Extends DTI replication across FA, MD, AD, and RD globally and locally, using the cleaned legacy metric tables when present.

In [ ]:
multi_out = run_section(
    "06_multimetric_dti",
    analysis_dti,
    lambda: run_multimetric_dti_analysis(paths, master, top_n=TOP_N_ROIS),
)
if multi_out is None:
    multi_out = run_section(
        "06b_live_multimetric_microstructure",
        analysis_dti,
        lambda: run_live_multimetric_overview(paths, master, global_micro=rd_out, node_micro=local_rd_out),
    )
maybe_display(multi_out, "summary")


## 7. Connectome QC and Availability

Confirms which connectome weighting schemes are available and which subjects contribute to each downstream network analysis.

In [ ]:
conn_qc_out = run_section(
    "07_connectome_qc_availability",
    analysis_graph,
    lambda: run_connectome_qc(paths, master),
)
maybe_display(conn_qc_out, "summary")


## 8. Global Graph Metrics

Replicates global strength, global efficiency, and characteristic path length analyses with robust statistics and permutation sensitivity.

In [ ]:
global_graph_out = run_section(
    "08_global_graph_metrics",
    analysis_graph,
    lambda: run_global_graph_metrics(paths, master),
)
if global_graph_out is None:
    global_graph_out = run_section(
        "08b_live_global_graph_metrics",
        analysis_graph,
        lambda: run_live_global_graph_metrics(paths, master),
    )
maybe_display(global_graph_out, "summary")


## 9. Node-wise Metrics and Centrality

Retains node-wise graph analysis and centrality review, with FDR control within each metric family.

In [ ]:
node_out = run_section(
    "09_nodewise_metrics",
    analysis_graph,
    lambda: run_nodewise_metrics(paths, master, top_n=TOP_N_ROIS),
)
if node_out is None:
    node_out = run_section(
        "09b_live_nodewise_graph_metrics",
        analysis_graph,
        lambda: run_live_nodewise_graph_metrics(paths, master, top_n=TOP_N_ROIS),
    )
centrality_out = run_section(
    "09c_centrality_review",
    analysis_graph,
    lambda: run_centrality_review(paths, master, top_n=TOP_N_ROIS),
)
if centrality_out is None:
    centrality_out = node_out
maybe_display(centrality_out, "summary")


## 10. Network-aware Microstructure and Coupling

Tests subject-level structure-microstructure coupling across nodes. This is structural coupling, not functional connectivity coupling.

In [ ]:
coupling_out = run_section(
    "10_structure_microstructure_coupling",
    analysis_coupling,
    lambda: run_coupling_analysis(paths, master),
)
if coupling_out is None:
    coupling_out = run_section(
        "10b_live_structure_microstructure_coupling",
        analysis_coupling,
        lambda: run_live_structure_microstructure_coupling(
            paths,
            master,
            node_graph=node_out if isinstance(node_out, dict) and "node_metrics" in node_out else None,
            node_micro=local_rd_out if isinstance(local_rd_out, dict) and "table" in local_rd_out else None,
        ),
    )
if coupling_out is not None and "summary" in coupling_out:
    display(coupling_out["summary"].sort_values("kw_p"))


## 11. Edge-wise Inference

Runs connectome-wide edge tests on the primary structural weighting, including Kruskal-Wallis omnibus testing and NBS-like pairwise component inference.

In [ ]:
edge_out = run_section(
    "11_edgewise_inference",
    analysis_edges,
    lambda: run_edgewise_inference(
        paths,
        master,
        connectome_type=EDGEWISE_CONNECTOME_TYPE,
        n_perm=EDGEWISE_N_PERM,
    ),
)
maybe_display(edge_out, "components", head=20)


## 12. Brain-age and Normative Modeling

Runs the structural brain-age model when legacy graph and DTI feature tables are available. The reported phenotype is age-corrected BAG.

In [ ]:
brain_age_out = run_section(
    "12_brain_age_normative_modeling",
    analysis_brain_age,
    lambda: run_brain_age_analysis(paths, master),
)
if brain_age_out is None:
    brain_age_out = run_section(
        "12b_live_brain_age_normative_modeling",
        analysis_brain_age,
        lambda: run_live_brain_age(paths, master),
    )
maybe_display(brain_age_out, "summary")


## 13. LR/SR Short- vs Long-Range Analysis

Derives CN-referenced short, medium, and long edge classes from `len_mean`, then compares short-range preservation and long-range degradation across groups.

In [ ]:
lrsr_out = run_section(
    "13_short_long_range_analysis",
    analysis_length_delay,
    lambda: run_lr_sr_analysis(
        paths,
        master,
        weight_type=PRIMARY_CONNECTOME,
        length_type=LENGTH_CONNECTOME,
        velocity_mm_per_ms=DELAY_VELOCITY_MM_PER_MS,
    ),
)
if lrsr_out is not None:
    display(pd.DataFrame([lrsr_out["thresholds"].__dict__]))
    display(lrsr_out["summary"])


## 14. Delay-informed Analyses

Computes a length-only delay proxy from structural path length and a fixed conduction velocity. Microstructure-informed delay remains conditional until compatible velocity/myelin inputs are available.

In [ ]:
print("Microstructure-informed delay inputs available:", microstructure_delay_available(paths))
delay_out = run_section(
    "14_delay_proxy_analysis",
    analysis_length_delay,
    lambda: run_delay_analysis(
        paths,
        master,
        weight_type=PRIMARY_CONNECTOME,
        length_type=LENGTH_CONNECTOME,
        velocity_mm_per_ms=DELAY_VELOCITY_MM_PER_MS,
    ),
)
maybe_display(delay_out, "summary")


## 15. Advanced Structural Innovation

Adds thesis-facing structural analyses: CN-referenced edge-distance relationship residuals, SR/LR structural compensation index, long-range vulnerability, and edge disease-gradient maps.

In [ ]:
advanced_out = run_section(
    "15_advanced_structural_innovation",
    analysis_advanced,
    lambda: run_structural_innovation_analysis(
        paths,
        master,
        weight_type=PRIMARY_CONNECTOME,
        length_type=LENGTH_CONNECTOME,
        top_n=TOP_N_ROIS,
    ),
)
maybe_display(advanced_out, "summary")
maybe_display(advanced_out, "edge_gradient", head=15)


## 16. Functional Coupling Placeholder

The thesis objective includes structural-functional decoupling, but this notebook does not fabricate FC results. This section records the required interface and stays pending until real rs-fMRI, EEG, or MEG connectivity matrices are supplied.

In [ ]:
functional_rows = [
    {
        "status": "pending_real_functional_connectivity",
        "required_inputs": "subject-level FC/EEG/MEG connectivity matrices aligned to AAL nodes and master subject_id",
        "planned_outputs": "SC-FC coupling, decoupling-vs-cognition models, FC-informed compensation tests",
        "current_policy": "disabled; structural-only analyses above are valid without FC claims",
    }
]
functional_placeholder = pd.DataFrame(functional_rows)
functional_out = run_section(
    "16_functional_coupling_placeholder",
    analysis_functional_placeholder,
    lambda: functional_placeholder,
)
maybe_display(functional_out)
functional_dir = paths.section_dir("16", "functional_placeholder")
functional_dir.mkdir(parents=True, exist_ok=True)
functional_placeholder.to_csv(functional_dir / "functional_coupling_requirements.csv", index=False)


## 17. Clinical Covariates and Sensitivity Analyses

Preserves legacy covariate and clinical-review outputs as sensitivity analyses, separate from the primary robust/permutation statistics.

In [ ]:
clinical_out = run_section(
    "17_clinical_covariate_sensitivity",
    analysis_clinical,
    lambda: run_clinical_covariate_review(paths),
)
maybe_display(clinical_out, "ancova", head=15)


## 18. Exports and Summary Tables

Collects completed section tables into a compact manifest and writes a multi-sheet Excel bundle for dissertation/manuscript review.

In [ ]:
def _result_table(result, key):
    return result.get(key) if isinstance(result, dict) and isinstance(result.get(key), pd.DataFrame) else None

summary_tables = {
    "analysis_snapshot": snapshot_df,
    "section_status": pd.DataFrame(SECTION_STATUS),
    "master_cohort": master,
    "qc_snapshot": qc_stage_counts,
}
for name, result, key in [
    ("qc_completeness", locals().get("qc_out"), "completeness"),
    ("demographics", locals().get("demo_out"), "counts"),
    ("global_rd", locals().get("rd_out"), "summary"),
    ("global_graph", locals().get("global_graph_out"), "summary"),
    ("coupling", locals().get("coupling_out"), "summary"),
    ("brain_age", locals().get("brain_age_out"), "summary"),
    ("lr_sr", locals().get("lrsr_out"), "summary"),
    ("delay", locals().get("delay_out"), "summary"),
    ("advanced_structural", locals().get("advanced_out"), "summary"),
]:
    table = _result_table(result, key)
    if table is not None:
        summary_tables[name] = table

section_status_df = pd.DataFrame(SECTION_STATUS)
section_status_df.to_csv(paths.exports_dir / "section_status.csv", index=False)
summary_xlsx = write_excel_bundle(summary_tables, paths.exports_dir / f"cohort_analysis_summary_{ANALYSIS_SNAPSHOT_MODE}.xlsx")
manifest = collect_csv_exports(paths.output_root)
manifest_path = paths.exports_dir / "export_manifest.csv"
manifest.to_csv(manifest_path, index=False)
display(section_status_df)
display(manifest.head(25))
summary_xlsx, manifest_path


## 19. Appendix / Support

Records the restored notebook source and working output roots for reproducibility.

In [ ]:
appendix_items = pd.DataFrame(
    [
        {"item": "Working notebook", "path": str(PROJECT_ROOT / "notebooks" / "structural_connectome_C.ipynb")},
        {"item": "Archived pre-EC2 notebook", "path": str(PROJECT_ROOT / "archive" / "structural_connectome_C_pre_ec2_port.ipynb")},
        {"item": "SOTA presentation", "path": str(PROJECT_ROOT / "docs" / "literature" / "SOTA_V_6.0.pptx")},
        {"item": "Legacy analysis root", "path": str(paths.legacy_analysis_dir)},
        {"item": "Revised analysis root", "path": str(paths.output_root)},
        {"item": "Snapshot mode", "path": ANALYSIS_SNAPSHOT_MODE},
    ]
)
display(appendix_items)


## 20. Complete Visualization Gallery

This final gallery displays every PNG generated under the Connectome C analysis output tree, grouped by section. These are the visual companions to the statistical tables and inference notes above.

In [ ]:
all_pngs = sorted(paths.output_root.rglob("*.png"))
print(f"Total generated PNG figures: {len(all_pngs)}")
current_section = None
for png_path in all_pngs:
    section = png_path.parent.relative_to(paths.output_root).as_posix()
    if section != current_section:
        current_section = section
        display(Markdown(f"### {section}"))
    display(Markdown(f"**{png_path.name}**"))
    display(Image(filename=str(png_path)))
